# NotifyOps AI: Clasificacion binaria de eventos sociales

Este notebook adapta el ejemplo de deteccion de spam entregado por el profesor al caso **NotifyOps**.

En el ejemplo original se clasifica un mensaje como `spam` o `no_spam`. En nuestro proyecto clasificamos eventos de una red social como:

- `0 = evento valido`
- `1 = evento riesgoso`

El modelo funciona como un filtro inteligente predictivo para apoyar la validacion del pipeline DataOps.


## 1. Importar librerias

Usamos `pandas`, `numpy` y `matplotlib`. El modelo de clasificacion logistica esta implementado en `src.notifyops_ai.modeling` para que funcione sin depender de scikit-learn.


In [ ]:
import pandas as pd
from pathlib import Path

from src.notifyops_ai import modeling


## 2. Crear dataset de eventos NotifyOps

El dataset simula eventos sociales de una red social: likes, comentarios, seguidores y tambien eventos anomalos.

Incluye casos como:

- tipo de evento fuera del alcance;
- usuario destino vacio;
- fecha invalida;
- eventos duplicados;
- riesgo historico por feedback.


In [ ]:
events = modeling.generate_synthetic_events(rows=320, seed=42)
events.head(10)


## 3. Analisis de calidad de datos

Esta etapa permite revisar cuantas filas existen, cuantos eventos son validos y cuantos presentan problemas. Es equivalente al analisis de nulos, duplicados y calidad pedido en la evaluacion.


In [ ]:
quality = modeling.quality_summary(events)
quality


## 4. Ingenieria de variables

Convertimos los eventos en variables numericas para que el modelo pueda aprender.

Ejemplos:

- `has_target_user`: si existe usuario destino;
- `has_valid_date`: si la fecha es valida;
- `is_duplicate`: si el evento esta duplicado;
- `event_type_invalid`: si el tipo de evento no pertenece a like/comment/follow.


In [ ]:
engineered = modeling.engineer_features(events)
engineered[modeling.FEATURE_COLUMNS + ['label_risky_event']].head()


## 5. Separar variables predictoras y variable objetivo

- `X`: variables del evento que el modelo usa para aprender.
- `y`: etiqueta real, donde `1` significa evento riesgoso y `0` evento valido.


In [ ]:
X = engineered[modeling.FEATURE_COLUMNS]
y = engineered['label_risky_event']

X_train, X_test, y_train, y_test = modeling.stratified_train_test_split(X, y, test_size=0.30, seed=42)
print('Entrenamiento:', len(X_train))
print('Prueba:', len(X_test))
print('Distribucion train:', y_train.value_counts().to_dict())
print('Distribucion test:', y_test.value_counts().to_dict())


## 6. Entrenar modelo de clasificacion

Usamos clasificacion logistica como modelo base, siguiendo la logica del notebook de spam.

La diferencia es que aqui no clasificamos texto, sino eventos sociales usando variables numericas y binarias.


In [ ]:
x_train_scaled, x_test_scaled, mean, std = modeling.standardize_train_test(X_train, X_test)
weights = modeling.train_logistic_regression(x_train_scaled, y_train.to_numpy(dtype=int))
probabilities = modeling.predict_probabilities(x_test_scaled, weights)
metrics, matrix = modeling.classification_metrics(y_test, probabilities)
metrics


## 7. Matriz de confusion

La matriz de confusion muestra aciertos y errores:

- evento valido detectado como valido;
- evento valido marcado como riesgoso;
- evento riesgoso marcado como valido;
- evento riesgoso detectado correctamente.

El error mas delicado es marcar un evento riesgoso como valido, porque podria generar una notificacion incorrecta.


In [ ]:
pd.DataFrame(matrix, index=['real_valido','real_riesgoso'], columns=['pred_valido','pred_riesgoso'])


## 8. Curva ROC, AUC y Gini

La curva ROC mide que tan bien el modelo separa eventos validos y riesgosos.

El indicador Gini se calcula como:

`Gini = 2 * AUC - 1`


In [ ]:
fpr, tpr, auc = modeling.roc_curve_values(y_test, probabilities)
gini = 2 * auc - 1
print('AUC:', round(auc, 4))
print('Gini:', round(gini, 4))


## 9. Ejecutar pipeline completo y guardar evidencias

Este comando genera dataset, metricas, graficos, predicciones nuevas, modelo guardado y dashboard HTML.


In [ ]:
result = modeling.run_ai_pipeline(rows=320, seed=42, save_plots=True)
result.metrics


## 10. Probar eventos nuevos

Igual que el notebook de spam prueba mensajes nuevos, aqui probamos eventos sociales nuevos.


In [ ]:
pd.read_csv('data/reports/ai/new_event_predictions.csv')


## 11. Conclusion

Este notebook demuestra que NotifyOps puede incorporar una capa de IA como filtro inteligente. El modelo no reemplaza la validacion por reglas; la complementa para anticipar eventos riesgosos y mejorar la calidad de las notificaciones.
